đánh giá 7 nhóm câu hỏi

metrics: intent accuracy / source relevance / keyword presence / latency



In [21]:
import os, sys, time

ROOT = os.path.dirname(os.path.abspath(""))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

os.environ["TQDM_DISABLE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

SEP  = "=" * 70
SEP2 = "-" * 50

# ground truth
TEST_CASES = [
    {
        "group": 1, "label": "Tra cứu dinh dưỡng",
        "query": "100g ức gà có bao nhiêu protein?",
        "expected_intent": "NUTRITION_LOOKUP",
        "expected_source_keywords": ["USDA"],
        "expected_answer_keywords": ["protein", "14"],
        "keyword_match": "all",
    },
    {
        "group": 2, "label": "Bệnh lý + chế độ ăn",
        "query": "Bị tiểu đường nên ăn gì và kiêng gì?",
        "expected_intent": "BOTH",
        "expected_source_keywords": ["Vinmec", "tiểu đường", "đường huyết", "đái tháo"],
        "expected_answer_keywords": ["tiểu đường", "chất xơ", "hạn chế", "rau", "đường huyết"],
        "keyword_match": "any",
    },
    {
        "group": 3, "label": "Triệu chứng",
        "query": "Bị đau đầu chóng mặt mệt mỏi nên bổ sung gì?",
        "expected_intent": "HEALTH_ADVICE",
        "expected_source_keywords": ["dinh dưỡng", "Vinmec", "thiếu máu"],
        "expected_answer_keywords": ["sắt", "magiê", "vitamin", "thiếu"],
        "keyword_match": "any",
    },
    {
        "group": 4, "label": "Mục tiêu cơ thể (gym/giảm cân)",
        "query": "Sau khi tập gym nên ăn gì để tăng cơ?",
        "expected_intent": "BOTH",
        "expected_source_keywords": ["Vinmec", "thể thao", "gym", "cơ"],
        "expected_answer_keywords": ["protein", "cơ", "thịt", "trứng", "sữa"],
        "keyword_match": "any",
    },
    {
        "group": 5, "label": "Theo đối tượng (bà bầu)",
        "query": "Bà bầu 3 tháng đầu nên bổ sung gì?",
        "expected_intent": "HEALTH_ADVICE",
        "expected_source_keywords": ["Vinmec", "thai", "bầu", "mang thai"],
        "expected_answer_keywords": ["acid folic", "folate", "sắt", "canxi", "rau", "vitamin"],
        "keyword_match": "any",
    },
    {
        "group": 6, "label": "Ăn chay / ăn kiêng",
        "query": "Ăn chay có đủ protein không, bổ sung từ đâu?",
        "expected_intent": "BOTH",
        "expected_source_keywords": ["Vinmec", "chay", "protein", "thực vật"],
        "expected_answer_keywords": ["protein", "đậu", "hạt", "đủ"],
        "keyword_match": "any",
    },
    {
        "group": 7, "label": "Câu phức hợp (cả 2 nhánh)",
        "query": "Người bị tiểu đường muốn giảm cân nên ăn gì?",
        "expected_intent": "BOTH",
        "expected_source_keywords": ["Vinmec", "tiểu đường", "đường huyết"],
        "expected_answer_keywords": ["tiểu đường", "giảm cân", "chất xơ", "rau", "hạn chế"],
        "keyword_match": "any",
    },
]

_results_log = []

def _check_intent(tc, result):
    return result.get("query_type", "") == tc["expected_intent"]

def _check_source(tc, result):
    sources_str = " ".join(result.get("sources", [])).lower()
    return any(kw.lower() in sources_str for kw in tc["expected_source_keywords"])

def _check_keywords(tc, result):
    answer_lower = result.get("answer", "").lower()
    kws = tc["expected_answer_keywords"]
    if tc.get("keyword_match") == "all":
        return all(kw.lower() in answer_lower for kw in kws)
    return any(kw.lower() in answer_lower for kw in kws)

def run_group(tc):
    t0 = time.time()
    result = pipeline.answer(tc["query"])
    latency = time.time() - t0

    ok_intent   = _check_intent(tc, result)
    ok_source   = _check_source(tc, result)
    ok_keywords = _check_keywords(tc, result)

    _results_log.append({
        "group": tc["group"], "latency": latency,
        "ok_intent": ok_intent, "ok_source": ok_source,
        "ok_keywords": ok_keywords,
        "has_answer": bool(result.get("answer", "").strip()),
    })

    print(SEP)
    print(f"NHÓM {tc['group']} — {tc['label']}")
    print(f"Câu hỏi : {tc['query']}")
    print(SEP2)

    actual_intent = result.get("query_type", "?")
    print(f"Intent   : {actual_intent}  [{'đúng' if ok_intent else 'sai'} expected: {tc['expected_intent']}]")

    ents = result.get("entities", {})
    if ents:
        for etype, vals in ents.items():
            if vals:
                print(f"  {etype:10s}: {', '.join(vals)}")
    else:
        print("  (no entity — keyword fallback)")

    nd = result.get("nutrition_data")
    if nd:
        print(f"\nUSDA     : {nd.get('food_description')} | "
              f"{nd.get('nutrient_name')} = "
              f"{nd.get('amount_per_100g')} {nd.get('unit')}/100g")

    sources = result.get("sources", [])
    print(f"\nNguồn    : {', '.join(sources[:3]) if sources else '(none)'}  [{'đúng' if ok_source else 'sai'}]")

    print(f"\nTrả lời  :\n{result.get('answer', '')}")
    print(f"\nUsed LLM : {result.get('used_llm', False)}")
    print(f"Latency  : {latency:.1f}s")
    print(f"Keywords : [{'đúng' if ok_keywords else 'sai'}] '{', '.join(tc['expected_answer_keywords'])}'")
    print(SEP)

print("setup done")

setup done


In [22]:
import warnings
warnings.filterwarnings("ignore")

from src.pipeline import NutritionPipeline
pipeline = NutritionPipeline()
print("pipeline ready")

[NERModel] Đã load từ d:\FoodRecomendationSystem\models/ner_phobert/phobert-ner-final — device: cpu
pipeline ready


In [23]:
# Cell 3 — Nhóm 1: Tra cứu dinh dưỡng
run_group(TEST_CASES[0])

NHÓM 1 — Tra cứu dinh dưỡng
Câu hỏi : 100g ức gà có bao nhiêu protein?
--------------------------------------------------
Intent   : NUTRITION_LOOKUP  [đúng expected: NUTRITION_LOOKUP]
  FOOD      : ức, gà
  NUTRIENT  : protein

USDA     : Chicken breast, roll, oven-roasted | Protein = 14.59 G/100g

Nguồn    : USDA FoodData Central (fdc_id=174608)  [đúng]

Trả lời  :
Câu trả lời trực tiếp: 100g ức gà chứa khoảng 14.59 gram protein.

Giải thích lý do hoặc cơ chế: Giá trị này được xác định dựa trên dữ liệu từ USDA FoodData Central và có thể thay đổi tùy thuộc vào phương pháp nấu nướng, nguồn gốc và chất lượng của thực phẩm.

Gợi ý thực phẩm hoặc lưu ý thực tế: Để tăng cường hấp thụ protein, bạn có thể kết hợp ức gà với các loại rau củ giàu vitamin và khoáng chất.

Used LLM : True
Latency  : 10.0s
Keywords : [đúng] 'protein, 14'


In [24]:
# Cell 4 — Nhóm 2: Bệnh lý + chế độ ăn
run_group(TEST_CASES[1])

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NHÓM 2 — Bệnh lý + chế độ ăn
Câu hỏi : Bị tiểu đường nên ăn gì và kiêng gì?
--------------------------------------------------
Intent   : BOTH  [đúng expected: BOTH]
  DISEASE   : tiểu_đường

Nguồn    : Vinmec  [đúng]

Trả lời  :
Bị tiểu đường nên ăn nhiều thực phẩm giàu chất xơ như rau củ quả, ngũ cốc nguyên hạt, và hoa quả. Những loại thực phẩm này giúp kiểm soát lượng đường trong máu và giảm nguy cơ mắc bệnh thận.

Giải thích lý do: Thực phẩm giàu chất xơ có thể giúp giảm tốc độ hấp thụ glucose vào máu, từ đó giúp kiểm soát lượng đường trong máu. Hơn nữa, chất xơ cũng có thể giúp giảm huyết áp và ngăn ngừa bệnh thận trở lên tồi tệ.

Gợi ý thực phẩm: Rau củ quả như bắp cải, cà rốt, ớt chuông; ngũ cốc nguyên hạt như gạo lứt, bánh mì nguyên cám; hoa quả như táo, cam, chanh.

Used LLM : True
Latency  : 12.7s
Keywords : [đúng] 'tiểu đường, chất xơ, hạn chế, rau, đường huyết'


In [25]:
# Cell 5 — Nhóm 3: Triệu chứng
run_group(TEST_CASES[2])

NHÓM 3 — Triệu chứng
Câu hỏi : Bị đau đầu chóng mặt mệt mỏi nên bổ sung gì?
--------------------------------------------------
Intent   : HEALTH_ADVICE  [đúng expected: HEALTH_ADVICE]
  SYMPTOM   : đau_đầu, chóng_mặt, mệt_mỏi

Nguồn    : Tổng hợp dinh dưỡng  [đúng]

Trả lời  :
Bị đau đầu, chóng mặt và mệt mỏi có thể do thiếu sắt và thiếu máu. Sắt là chất dinh dưỡng cần thiết để tạo hemoglobin vận chuyển oxy đến não và cơ bắp.

Lý do: Thiếu sắt sẽ dẫn đến thiếu máu, khiến não không nhận đủ oxy, gây ra các triệu chứng như đau đầu, chóng mặt, mệt mỏi và da xanh xao.

Gợi ý thực phẩm bổ sung: Bổ sung sắt bằng cách tiêu thụ thịt đỏ, gan, sò huyết và rau dền.

Used LLM : True
Latency  : 5.4s
Keywords : [đúng] 'sắt, magiê, vitamin, thiếu'


In [26]:
# Cell 6 — Nhóm 4: Mục tiêu cơ thể (gym)
run_group(TEST_CASES[3])

NHÓM 4 — Mục tiêu cơ thể (gym/giảm cân)
Câu hỏi : Sau khi tập gym nên ăn gì để tăng cơ?
--------------------------------------------------
Intent   : BOTH  [đúng expected: BOTH]
  (no entity — keyword fallback)

Nguồn    : Vinmec  [đúng]

Trả lời  :
Sau khi tập gym, bạn nên ăn những thực phẩm giàu protein và carbohydrate để hỗ trợ quá trình xây dựng cơ bắp và phục hồi sau khi tập luyện. Các thực phẩm như thịt nạc, cá, trứng, sữa, ngũ cốc, trái cây và rau củ đều là lựa chọn tốt.

Lý do: Protein giúp xây dựng và sửa chữa cơ bắp, trong khi carbohydrate cung cấp năng lượng cho cơ thể. Sau khi tập gym, cơ thể cần thời gian để phục hồi và tái tạo các mô bị tổn thương, và thực phẩm giàu protein và carbohydrate sẽ hỗ trợ quá trình này.

Gợi ý thực phẩm: Thịt nạc (như gà hoặc bò), cá (như cá hồi hoặc cá ngừ), trứng, sữa, ngũ cốc (như bánh mì hoặc gạo), trái cây (như táo hoặc chuối) và rau củ (như cà rốt hoặc bông cải xanh).

Nguồn: [Vinmec] Bổ sung vitamin tốt cho người tập gym bằng cách nào?



In [27]:
# Cell 7 — Nhóm 5: Theo đối tượng (bà bầu)
run_group(TEST_CASES[4])

NHÓM 5 — Theo đối tượng (bà bầu)
Câu hỏi : Bà bầu 3 tháng đầu nên bổ sung gì?
--------------------------------------------------
Intent   : HEALTH_ADVICE  [đúng expected: HEALTH_ADVICE]
  (no entity — keyword fallback)

Nguồn    : Vinmec  [đúng]

Trả lời  :
Bà bầu 3 tháng đầu nên bổ sung rau lá xanh như rau xà lách, cải xoăn, rau bina, bông cải xanh, cải ngọt, lá mù tạt... để cung cấp chất dinh dưỡng cho bản thân và phù hợp cho sự phát triển của em bé.

Lý do: Rau lá xanh chứa nhiều vitamin, khoáng chất và chất chống oxy hóa giúp hỗ trợ quá trình sinh trưởng và phát triển của thai nhi. Chúng cũng có thể giúp giảm nguy cơ thiếu máu và các vấn đề về sức khỏe khác ở mẹ bầu.

Gợi ý thực phẩm: Ngoài rau lá xanh, bà bầu 3 tháng đầu nên bổ sung thêm trái cây tươi, ngũ cốc nguyên hạt, protein từ nguồn động vật hoặc thực vật để đảm bảo cung cấp đủ chất dinh dưỡng cho bản thân và thai nhi.

Used LLM : True
Latency  : 6.5s
Keywords : [đúng] 'acid folic, folate, sắt, canxi, rau, vitamin'


In [28]:
# Cell 8 — Nhóm 6: Ăn chay / ăn kiêng
run_group(TEST_CASES[5])

NHÓM 6 — Ăn chay / ăn kiêng
Câu hỏi : Ăn chay có đủ protein không, bổ sung từ đâu?
--------------------------------------------------
Intent   : BOTH  [đúng expected: BOTH]
  NUTRIENT  : protein

Nguồn    : Vinmec  [đúng]

Trả lời  :
Ăn chay vẫn có thể cung cấp đủ protein nếu được thực hiện một cách hợp lý. Protein cần thiết cho cơ thể có thể được lấy từ các loại thực phẩm như đậu nành, đậu xanh, đậu đỏ, hạt điều, hạt hướng dương, các loại hạt khác, các loại đậu và các loại rau giàu protein.

Giải thích: Cơ thể con người cần khoảng 0,8-1 gram protein mỗi ngày cho mỗi kilogram trọng lượng cơ thể. Do đó, với một người nặng 60 kg, cần khoảng 48-60 gram protein mỗi ngày. Các loại thực phẩm như đậu nành, đậu xanh, đậu đỏ, hạt điều, hạt hướng dương, các loại hạt khác, các loại đậu và các loại rau giàu protein đều có thể cung cấp đủ protein cho cơ thể.

Gợi ý: Người ăn chay nên đa dạng hóa chế độ ăn uống của mình để đảm bảo cung cấp đủ protein. Ngoài ra, cũng cần lưu ý rằng một số thực phẩm n

In [29]:
# Cell 9 — Nhóm 7: Câu phức hợp (cả 2 nhánh)
run_group(TEST_CASES[6])

NHÓM 7 — Câu phức hợp (cả 2 nhánh)
Câu hỏi : Người bị tiểu đường muốn giảm cân nên ăn gì?
--------------------------------------------------
Intent   : BOTH  [đúng expected: BOTH]
  DISEASE   : tiểu_đường

Nguồn    : Vinmec  [đúng]

Trả lời  :
Người bị tiểu đường muốn giảm cân nên ăn thực phẩm ít calo, nhiều chất xơ và protein, như rau củ quả, ngũ cốc nguyên hạt, cá béo, trứng. Ngoài ra, họ cũng nên hạn chế hoặc tránh các loại thực phẩm giàu đường, muối và chất béo.

Giải thích: Thực phẩm ít calo giúp giảm cân hiệu quả vì chúng không chứa nhiều năng lượng. Chất xơ giúp ngăn chặn sự hấp thụ đường vào máu, giảm nguy cơ tăng đường huyết. Protein giúp duy trì cảm giác no, giảm thèm ăn và hỗ trợ quá trình trao đổi chất.

Lưu ý thực tế: Người bị tiểu đường nên tham khảo ý kiến của bác sĩ hoặc chuyên gia dinh dưỡng để xây dựng một kế hoạch ăn uống phù hợp với tình trạng sức khỏe của mình.

Used LLM : True
Latency  : 6.3s
Keywords : [đúng] 'tiểu đường, giảm cân, chất xơ, rau, hạn chế'


In [30]:
n = len(_results_log)
if n == 0:
    print("chưa chạy nhóm nào.")
else:
    ok_intent   = sum(r["ok_intent"]   for r in _results_log)
    ok_source   = sum(r["ok_source"]   for r in _results_log)
    ok_keywords = sum(r["ok_keywords"] for r in _results_log)
    has_answer  = sum(r["has_answer"]  for r in _results_log)
    lats        = [r["latency"] for r in _results_log]

    print("=" * 50)
    print("SUMMARY REPORT")
    print("-" * 50)
    print(f"  Nhóm đã chạy         : {n}/7")
    print(f"  Answer (non-empty)   : {has_answer}/{n}  ({has_answer/n*100:.1f}%)")
    print(f"  Intent accuracy      : {ok_intent}/{n}  ({ok_intent/n*100:.1f}%)")
    print(f"  Source relevance     : {ok_source}/{n}  ({ok_source/n*100:.1f}%)")
    print(f"  Keyword presence     : {ok_keywords}/{n}  ({ok_keywords/n*100:.1f}%)")
    print(f"  Latency avg          : {sum(lats)/n:.1f}s  |  min {min(lats):.1f}s  |  max {max(lats):.1f}s")
    print("=" * 50)

    print("\nChi tiết:")
    print(f"  {'Nhóm':<8} {'Intent':>8} {'Source':>8} {'Keywords':>10} {'Latency':>10}")
    print("  " + "-" * 46)
    for r in _results_log:
        print(f"  {r['group']:<8} "
              f"{'đúng' if r['ok_intent'] else 'sai':>8} "
              f"{'đúng' if r['ok_source'] else 'sai':>8} "
              f"{'đúng' if r['ok_keywords'] else 'sai':>10} "
              f"{r['latency']:>9.1f}s")

SUMMARY REPORT
--------------------------------------------------
  Nhóm đã chạy         : 7/7
  Answer (non-empty)   : 7/7  (100.0%)
  Intent accuracy      : 7/7  (100.0%)
  Source relevance     : 7/7  (100.0%)
  Keyword presence     : 7/7  (100.0%)
  Latency avg          : 8.0s  |  min 5.4s  |  max 12.7s

Chi tiết:
  Nhóm       Intent   Source   Keywords    Latency
  ----------------------------------------------
  1            đúng     đúng       đúng      10.0s
  2            đúng     đúng       đúng      12.7s
  3            đúng     đúng       đúng       5.4s
  4            đúng     đúng       đúng       7.3s
  5            đúng     đúng       đúng       6.5s
  6            đúng     đúng       đúng       8.1s
  7            đúng     đúng       đúng       6.3s
